# Reading Satellite Imagery with GeoTIFFs

This notebook walks through loading a **Sentinel-2** satellite image from Microsoft's Planetary Computer, reading a small window of data, and visualizing the difference between **Red** and **Near-Infrared (NIR)** bands.

**Key concepts:**
- GeoTIFF files store raster data with geographic metadata
- Sentinel-2 L2A provides surface reflectance across 13 spectral bands
- Healthy vegetation **absorbs** red light and **reflects** NIR light

In [6]:
import rasterio
import planetary_computer as pc
from pystac_client import Client

import numpy as np
import matplotlib.pyplot as plt

## 1. Connect to the Data Catalog

[Planetary Computer](https://planetarycomputer.microsoft.com/) hosts petabytes of free, analysis-ready satellite data. The `pystac_client` library lets us search the catalog, and `planetary_computer` handles URL signing so we can read the files directly.

In [ ]:
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

print("Connected to Planetary Computer")

## 2. Search for Imagery

We search for a Sentinel-2 L2A scene over **Jasper County, Iowa** (prime corn country) during **July 2023** — peak growing season.

In [ ]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-93.107, 41.868, -93.087, 41.888],
    datetime="2023-07-01/2023-07-31",
    max_items=1,
)

## 3. Get the Image Asset

Each search result (`item`) contains multiple assets — one per spectral band. We grab **Band 4 (Red)** first.

In [21]:
item = list(search.items())[0]
url = item.assets["B04"].href
print("Got URL:", url[:60], "...")

Got URL: https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/15 ...


In [ ]:
b04_url = item.assets["B04"].href

print("Band 4 URL starts with:")
print(b04_url[:80], "...")

## 4. Inspect the GeoTIFF

Before reading pixel data, let's check the file's metadata: image size, coordinate system, and data type.

In [23]:
with rasterio.open(b04_url) as src:
    print("Image shape:      ", src.shape)
    print("Number of bands:  ", src.count)
    print("Coordinate system:", src.crs)
    print("Data type:        ", src.dtypes[0])

Image shape:       (10980, 10980)
Number of bands:   1
Coordinate system: EPSG:32615
Data type:         uint16


## 5. Read a Small Window

The full tile is 10,980 x 10,980 pixels — too large to load into memory at once. We use `rasterio.windows.Window` to read just a **256 x 256** patch from the center of the tile.

In [ ]:
with rasterio.open(b04_url) as src:
    window = rasterio.windows.Window(
        col_off=5000,
        row_off=5000,
        width=256,
        height=256,
    )
    data = src.read(1, window=window)

print("Array shape:", data.shape)
print("Data type:  ", data.dtype)
print("Min value:  ", data.min())
print("Max value:  ", data.max())

## 6. Convert to Surface Reflectance

Sentinel-2 L2A stores reflectance as integers scaled by **10,000**. Dividing by 10,000 gives us physically meaningful values between 0 and 1, where:
- **0** = no light reflected
- **1** = all light reflected

In [ ]:
reflectance = data.astype(np.float32) / 10000.0

print("Min reflectance: ", reflectance.min().round(4))
print("Max reflectance: ", reflectance.max().round(4))
print("Mean reflectance:", reflectance.mean().round(4))

## 7. Visualize the Red Band

> **Why `vmax=0.3`?** By default, matplotlib auto-scales to the brightest pixel, which washes out contrast in dark areas. Setting `vmax=0.3` treats 0.3 reflectance as white — anything above clips to white. This makes vegetation patterns much easier to see. We'll use this technique throughout the project.

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(
    reflectance,
    cmap='gray',
    vmin=0,
    vmax=0.3,
)

plt.title('Band 4 (Red) — Iowa, July 2023')
plt.colorbar(label='Reflectance')
plt.axis('off')
plt.tight_layout()
plt.show()

## 8. Load the NIR Band

Now we load **Band 8 (Near-Infrared)** from the same image — same location, same moment in time, just a different wavelength. Healthy vegetation reflects strongly in the NIR, so we expect much higher reflectance values than the Red band.

In [ ]:
b08_url = item.assets["B08"].href

with rasterio.open(b08_url) as src:
    window = rasterio.windows.Window(5000, 5000, 256, 256)
    nir_data = src.read(1, window=window).astype(np.float32) / 10000.0

print("NIR mean reflectance:", nir_data.mean().round(4))
print("Red mean reflectance:", reflectance.mean().round(4))

## 9. Compare Red vs. NIR

Side-by-side comparison showing the spectral signature of vegetation:
- **Red band** — plants absorb red light for photosynthesis, so they appear **dark**
- **NIR band** — plants reflect NIR light (cell structure scattering), so they appear **bright**

This contrast is the foundation of vegetation indices like **NDVI**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(reflectance, cmap='gray', vmin=0, vmax=0.3)
axes[0].set_title('Band 4 — Red\n(plants look DARK — absorbing)')
axes[0].axis('off')

axes[1].imshow(nir_data, cmap='gray', vmin=0, vmax=0.5)
axes[1].set_title('Band 8 — Near-Infrared\n(plants look BRIGHT — reflecting)')
axes[1].axis('off')

plt.suptitle('Same field, same moment, different wavelengths', y=1.02)
plt.tight_layout()
plt.show()